In [ ]:
#Load libraries
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
# Load libraries
import sys
sys.path.append(r"e:\Coding Projects\Investment Analysis")
from Quantapp.visualization import Plotter
from Quantapp.data import MacroDataClient
from Quantapp.data import MarketDataClient
from Quantapp.data import get_historical_treasury_yields

import numpy as np
import json
import os
import pandas as pd
from Quantapp.data import yf as qa_yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import holidays
import plotly.express as px
import concurrent.futures
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#shut down warnings
import warnings
warnings.filterwarnings("ignore")



qp = Plotter()
qe = MacroDataClient()
md = MarketDataClient()

In [ ]:
#3A Historical Treasury Yields
treasury_start_date = None # Pull the full available FRED history for each maturity.
treasury_maturity_order = ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
broad_treasury_yields = pd.DataFrame()
ordered_maturities = []
broad_treasury_average_yield = pd.Series(dtype=float)

try:
    broad_treasury_yields = get_historical_treasury_yields(start_date=treasury_start_date).sort_index().dropna(how='all')
except ValueError as exc:
    print(f'Skipping Historical Treasury Yields: {exc}')
else:
    ordered_maturities = [maturity for maturity in treasury_maturity_order if maturity in broad_treasury_yields.columns]

    if not ordered_maturities:
        print('No Treasury yield history is available for plotting.')
    else:
        treasury_history_fig = go.Figure()
        for maturity in ordered_maturities:
            series = broad_treasury_yields[maturity].dropna()
            if series.empty:
                continue
            treasury_history_fig.add_trace(
                go.Scatter(
                    x=series.index,
                    y=series,
                    mode='lines',
                    name=maturity,
                    line=dict(width=1.6),
                )
            )

        treasury_history_fig.update_layout(
            title='Historical Treasury Yields',
            template='plotly_dark',
            height=700,
            hovermode='x unified',
            legend_title_text='Maturity',
            margin=dict(l=40, r=40, t=90, b=40),
        )
        treasury_history_fig.update_xaxes(title_text='Date')
        treasury_history_fig.update_yaxes(title_text='Yield (%)')
        treasury_history_fig.show(config={'responsive': True})

        broad_treasury_average_yield = broad_treasury_yields[ordered_maturities].mean(axis=1).dropna()
        if broad_treasury_average_yield.empty:
            print('No Treasury average-yield history is available for plotting.')
        else:
            average_yield_fig = go.Figure()
            average_yield_fig.add_trace(
                go.Scatter(
                    x=broad_treasury_average_yield.index,
                    y=broad_treasury_average_yield,
                    mode='lines',
                    name='Average Yield Level',
                    line=dict(color='#f59e0b', width=2.5),
                    hovertemplate='Date=%{x|%Y-%m-%d}<br>Average yield=%{y:.2f}%<extra></extra>',
                )
            )
            average_yield_fig.add_hline(
                y=float(broad_treasury_average_yield.mean()),
                line_dash='dot',
                line_color='rgba(245, 158, 11, 0.55)',
                annotation_text='Full-Sample Mean',
                annotation_position='top left',
            )
            average_yield_fig.update_layout(
                title='Historical Average Treasury Yield Across Maturities',
                template='plotly_dark',
                height=500,
                hovermode='x unified',
                legend_title_text='Series',
                margin=dict(l=40, r=40, t=90, b=40),
            )
            average_yield_fig.update_xaxes(title_text='Date')
            average_yield_fig.update_yaxes(title_text='Average Yield (%)')
            average_yield_fig.show(config={'responsive': True})


In [ ]:
#3B Treasury Yield Curve Animation
if broad_treasury_yields.empty or not ordered_maturities:
    print('Run the Historical Treasury Yields cell first to load Treasury data for the animation.')
else:
    treasury_curve_animation_data = broad_treasury_yields[ordered_maturities].ffill().dropna(how='all')
    monthly_frame_dates = (
        treasury_curve_animation_data.groupby(treasury_curve_animation_data.index.to_period('M'))
        .apply(lambda frame: frame.index[-1])
        .tolist()
    )
    treasury_curve_animation_data = treasury_curve_animation_data.loc[monthly_frame_dates]

    if treasury_curve_animation_data.empty:
        print('No Treasury yield curve snapshots are available for animation.')
    else:
        curve_min = float(treasury_curve_animation_data.min().min())
        curve_max = float(treasury_curve_animation_data.max().max())
        curve_padding = max(0.25, (curve_max - curve_min) * 0.08) if curve_max > curve_min else 0.5

        def _curve_trace(frame_date, values):
            return go.Scatter(
                x=ordered_maturities,
                y=values.tolist(),
                mode='lines+markers',
                line=dict(color='#60a5fa', width=3),
                marker=dict(color='#f8fafc', size=9, line=dict(color='#60a5fa', width=1.5)),
                fill='tozeroy',
                fillcolor='rgba(96, 165, 250, 0.18)',
                hovertemplate='Maturity=%{x}<br>Yield=%{y:.2f}%<extra></extra>',
                name=frame_date.strftime('%Y-%m-%d'),
            )

        def _average_level_trace(average_yield):
            return go.Scatter(
                x=ordered_maturities,
                y=[average_yield] * len(ordered_maturities),
                mode='lines',
                name='Average Curve Level',
                line=dict(color='#f59e0b', width=2, dash='dot'),
                hovertemplate='Average curve level=%{y:.2f}%<extra></extra>',
            )

        initial_curve_date = treasury_curve_animation_data.index[0]
        initial_curve_values = treasury_curve_animation_data.iloc[0]
        initial_curve_average = float(initial_curve_values.mean())
        animation_frames = []
        slider_steps = []

        for frame_date, curve_values in treasury_curve_animation_data.iterrows():
            frame_name = frame_date.strftime('%Y-%m-%d')
            curve_average = float(curve_values.mean())
            animation_frames.append(
                go.Frame(
                    name=frame_name,
                    data=[
                        _curve_trace(frame_date, curve_values),
                        _average_level_trace(curve_average),
                    ],
                )
            )
            slider_steps.append(
                dict(
                    label=frame_date.strftime('%Y-%m'),
                    method='animate',
                    args=[
                        [frame_name],
                        {
                            'mode': 'immediate',
                            'frame': {'duration': 0, 'redraw': True},
                            'transition': {'duration': 0},
                        },
                    ],
                )
            )

        treasury_curve_animation_fig = go.Figure(
            data=[
                _curve_trace(initial_curve_date, initial_curve_values),
                _average_level_trace(initial_curve_average),
            ],
            frames=animation_frames,
        )
        treasury_curve_animation_fig.update_layout(
            title='Treasury Yield Curve Animation',
            template='plotly_dark',
            height=650,
            hovermode='closest',
            margin=dict(l=40, r=40, t=90, b=110),
            xaxis=dict(
                title='Maturity',
                categoryorder='array',
                categoryarray=ordered_maturities,
            ),
            yaxis=dict(
                title='Yield (%)',
                range=[curve_min - curve_padding, curve_max + curve_padding],
            ),
            updatemenus=[
                dict(
                    type='buttons',
                    direction='left',
                    showactive=False,
                    x=0.01,
                    y=1.16,
                    xanchor='left',
                    yanchor='top',
                    buttons=[
                        dict(
                            label='Play',
                            method='animate',
                            args=[
                                None,
                                {
                                    'fromcurrent': True,
                                    'frame': {'duration': 220, 'redraw': True},
                                    'transition': {'duration': 140},
                                },
                            ],
                        ),
                        dict(
                            label='Pause',
                            method='animate',
                            args=[
                                [None],
                                {
                                    'mode': 'immediate',
                                    'frame': {'duration': 0, 'redraw': False},
                                    'transition': {'duration': 0},
                                },
                            ],
                        ),
                    ],
                )
            ],
            sliders=[
                dict(
                    active=0,
                    currentvalue={'prefix': 'Curve Date: '},
                    pad={'t': 55},
                    len=0.95,
                    x=0.03,
                    xanchor='left',
                    y=0,
                    yanchor='top',
                    steps=slider_steps,
                )
            ],
        )
        treasury_curve_animation_fig.show(config={'responsive': True})


In [ ]:
#3C Treasury Slopes and Belly
if broad_treasury_yields.empty or not ordered_maturities:
    print('Run the Historical Treasury Yields cell first to load Treasury data for the slope and belly plots.')
else:
    treasury_curve_filled = broad_treasury_yields[ordered_maturities].ffill().dropna(how='all')
    slope_series_map = {}
    slope_pairs = [
        ('10Y - 3M', '10Y', '3M'),
        ('10Y - 2Y', '10Y', '2Y'),
        ('5Y - 2Y', '5Y', '2Y'),
        ('30Y - 10Y', '30Y', '10Y'),
    ]

    for label, long_maturity, short_maturity in slope_pairs:
        if {long_maturity, short_maturity}.issubset(treasury_curve_filled.columns):
            slope_series_map[label] = (
                treasury_curve_filled[long_maturity] - treasury_curve_filled[short_maturity]
            ).dropna()

    belly_series_map = {}
    if {'2Y', '5Y', '10Y'}.issubset(treasury_curve_filled.columns):
        belly_series_map['5Y Belly vs Avg(2Y,10Y)'] = (
            treasury_curve_filled['5Y'] - (
                (treasury_curve_filled['2Y'] + treasury_curve_filled['10Y']) / 2
            )
        ).dropna()

    slope_df = pd.DataFrame(slope_series_map).dropna(how='all')
    belly_df = pd.DataFrame(belly_series_map).dropna(how='all')

    if slope_df.empty and belly_df.empty:
        print('No Treasury slope or belly series are available for plotting.')
    else:
        treasury_slope_fig = make_subplots(
            rows=2,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.1,
            subplot_titles=(
                'Historical Treasury Slopes',
                'Historical Belly / Curvature',
            ),
        )

        slope_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
        for idx, column in enumerate(slope_df.columns):
            treasury_slope_fig.add_trace(
                go.Scatter(
                    x=slope_df.index,
                    y=slope_df[column],
                    mode='lines',
                    name=column,
                    line=dict(color=slope_palette[idx % len(slope_palette)], width=2.2),
                ),
                row=1,
                col=1,
            )

        belly_palette = ['#f59e0b', '#22c55e', '#f97316']
        for idx, column in enumerate(belly_df.columns):
            treasury_slope_fig.add_trace(
                go.Scatter(
                    x=belly_df.index,
                    y=belly_df[column],
                    mode='lines',
                    name=column,
                    line=dict(color=belly_palette[idx % len(belly_palette)], width=2.4),
                ),
                row=2,
                col=1,
            )

        treasury_slope_fig.add_hline(y=0, line_dash='dot', line_color='rgba(226, 232, 240, 0.45)', row=1, col=1)
        treasury_slope_fig.add_hline(y=0, line_dash='dot', line_color='rgba(226, 232, 240, 0.45)', row=2, col=1)
        treasury_slope_fig.update_yaxes(title_text='Spread (percentage points)', row=1, col=1)
        treasury_slope_fig.update_yaxes(title_text='Belly Spread (%)', row=2, col=1)
        treasury_slope_fig.update_xaxes(title_text='Date', row=2, col=1)
        treasury_slope_fig.update_layout(
            title='Treasury Slopes and Belly Over Time',
            template='plotly_dark',
            height=850,
            hovermode='x unified',
            legend_title_text='Series',
            margin=dict(l=40, r=40, t=90, b=40),
        )
        treasury_slope_fig.show(config={'responsive': True})


In [ ]:
bond_market = md.get_bond_data()
us_govt = md.get_bond_data('US Government')
treasuries = md.get_bond_data('Treasuries')
corporate_bonds = md.get_bond_data('Corporate Bonds')
low_risk_credit = md.get_bond_data('low risk credit')
medium_risk_credit = md.get_bond_data('medium risk credit')
high_risk_credit = md.get_bond_data('high risk credit')
convertible_bonds = md.get_bond_data('convertible bonds')
preferred_stocks = md.get_bond_data('preferred stocks')
municipal_bonds = md.get_bond_data('municipal bonds')
structured_credit = md.get_bond_data('structured credit')
international_sovereign = md.get_bond_data('international/sovereign')
floating_rate = md.get_bond_data('floating rate')

'''
types = [
    'bond market',
    'US Government',
    'Treasuries',
    'Corporate Bonds',
    'low risk credit',
    'medium risk credit',
    'high risk credit',
    'convertible bonds',
    'preferred stocks',
    'municipal bonds',
    'structured credit',
    'international/sovereign',
    'floating rate'
]


'''

In [ ]:
def plot_market_data(market_dict, title):
    fig = make_subplots(rows=1, cols=1)
    for key in market_dict.keys():
        df = market_dict[key]
        fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name=key))
    fig.update_layout(title=title, xaxis_title='Date', yaxis_title='Close Price')
    fig.show()


In [ ]:
#plot bond market data
plot_market_data(bond_market, 'Bond Market ETFs')
plot_market_data(us_govt, 'US Government Bond ETFs')
plot_market_data(treasuries, 'Treasury ETFs')
plot_market_data(corporate_bonds, 'Corporate Bond ETFs')
plot_market_data(low_risk_credit, 'Low Risk Credit Bond ETFs')
plot_market_data(medium_risk_credit, 'Medium Risk Credit Bond ETFs')
plot_market_data(high_risk_credit, 'High Risk Credit Bond ETFs')
plot_market_data(convertible_bonds, 'Convertible Bond ETFs')
plot_market_data(preferred_stocks, 'Preferred Stock ETFs')
plot_market_data(municipal_bonds, 'Municipal Bond ETFs')
plot_market_data(structured_credit, 'Structured Credit ETFs')
plot_market_data(international_sovereign, 'International/Sovereign Bond ETFs')
plot_market_data(floating_rate, 'Floating Rate Bond ETFs')

